# Simplicits with Rigid Body Simulation

This notebook demonstrates coupling Simplicits soft-body simulation with Newton rigid body
dynamics. Two deformable cubes fall and interact with a rigid body cube driven by
`SolverSemiImplicit`, all sharing the same Newton model and collision pipeline.

## Imports

We import Kaolin, Newton, Warp, and **k3d** for interactive 3D visualization in the
notebook. `transform_points` from `newton._src.geometry.utils` is used to transform
rigid body mesh vertices by the body's pose each frame.

In [ ]:
import os
import threading

import k3d
import numpy as np
import torch
import warp as wp
from IPython.display import display
from ipywidgets import Button, VBox
from newton._src.geometry.utils import transform_points

import kaolin
from kaolin.physics.simplicits import SimplicitsObject
from examples.tutorial.tutorial_common import COMMON_DATA_DIR

import newton
from kaolin.experimental.newton.builder import SimplicitsModelBuilder
from kaolin.experimental.newton.solver import SimplicitsSolver
from newton.solvers import SolverSemiImplicit
from newton._src.sim.collide import CollisionPipeline

wp.clear_kernel_cache()

## Configuration

Key constants:
- **`MESH_SCALE`** — scales the loaded cube mesh before adding it as a Newton collision shape.
- **`SOFT_YOUNGS_MODULUS` / `POISSON_RATIO` / `DENSITY`** — elastic material parameters for the
  Simplicits soft-body cubes.
- **`DT`** — simulation timestep (seconds per frame).
- **`FLOOR_PLANE`** — y-coordinate of the ground plane.

In [ ]:
# Mesh parameters
MESH_SCALE = 1.0

# Physics material parameters
SOFT_YOUNGS_MODULUS = 1e5
POISSON_RATIO = 0.45
DENSITY = 500        # kg/m^3
APPROX_VOLUME = 0.5  # m^3
NUM_SAMPLES = 10000

# Simulation parameters
DT = 0.03
FLOOR_PLANE = -1.0
GRAVITY_ACC = 9.8

# Newton solver parameters
PENALTY_WEIGHT = 10000.0
MAX_NEWTON_STEPS = 10
MAX_LS_STEPS = 30
CONV_TOL = 1e-8
ELASTICITY_TYPE = "neohookean_elasticity"
NEWTON_HESSIAN_REGULARIZER = 1e-2
GRAVITY_ON = True

## Mesh Loading and Simplicits Object

We load a cube mesh and sample interior points for Simplicits. Material properties
(Young's modulus, Poisson ratio, density) are assigned per sample point.

`SimplicitsObject.create_rigid` creates a reduced-DOF elastic object with a single handle (for fast construction) in this example.

In [ ]:
def load_mesh():
    """Load and prepare the mesh for simulation."""
    mesh_path = os.path.join(COMMON_DATA_DIR, "meshes")
    mesh = kaolin.io.import_mesh(
        mesh_path + "/cube.obj", triangulate=True).to('cuda')
    mesh.vertices = kaolin.ops.pointcloud.center_points(
        mesh.vertices.unsqueeze(0), normalize=True).squeeze(0)
    return mesh


def create_simplicits_object(mesh):
    """Create a Simplicits object from mesh."""
    orig_vertices = mesh.vertices.clone()

    # Sample points uniformly over the bounding box
    uniform_pts = torch.rand(NUM_SAMPLES, 3, device='cuda') * (
        orig_vertices.max(dim=0).values - orig_vertices.min(dim=0).values
    ) + orig_vertices.min(dim=0).values

    # Create material property tensors
    yms = torch.full((NUM_SAMPLES,), SOFT_YOUNGS_MODULUS, device='cuda')
    prs = torch.full((NUM_SAMPLES,), POISSON_RATIO, device='cuda')
    rhos = torch.full((NUM_SAMPLES,), DENSITY, device='cuda')

    # Create single handle Simplicits object
    sim_obj = SimplicitsObject.create_rigid(
        uniform_pts, yms, prs, rhos, APPROX_VOLUME
    ) # create_rigid creates a 1 handle object that affinely deforms. Its cheap to construct and at higher stiffnesses, this acts as a rigid object.
    return sim_obj


mesh = load_mesh()
sim_obj = create_simplicits_object(mesh)
print(f"Mesh: {mesh.vertices.shape[0]} vertices | Sim samples: {sim_obj.pts.shape[0]}")

## Building the Newton Model

`SimplicitsModelBuilder` extends Newton's `ModelBuilder`. We:
1. Add two Simplicits objects (two soft cubes, the second offset upward by 3 m).
2. Add a ground plane.
3. Configure inter-object collision handling.
4. Add one rigid cube body (driven by `SolverSemiImplicit`).

`finalize()` compiles all geometry and physics parameters into GPU arrays.

In [ ]:
def add_cube_mesh(builder, mesh, position):
    """Add a rigid cube mesh body to the builder."""
    newton_mesh = newton.Mesh(
        mesh.vertices.detach().cpu().numpy() * MESH_SCALE,
        mesh.faces.detach().cpu().numpy()
    )
    body = builder.add_body(xform=wp.transform(
        wp.vec3(position[0], position[1], position[2]), wp.quat_identity()
    ))
    builder.add_joint_free(body)
    builder.add_shape_mesh(body=body, mesh=newton_mesh)


def create_builder_model(sim_obj, mesh):
    """Create and configure the Simplicits model builder."""
    print("Building model...")
    builder = SimplicitsModelBuilder(up_axis="y", gravity=-GRAVITY_ACC)

    # Add two Simplicits soft-body cubes; capture their object IDs
    builder.add_simplicits_object(sim_obj)
    builder.add_simplicits_object(
        sim_obj,
        init_transform=torch.tensor([
            [1, 0, 0, 0],
            [0, 1, 0, 3.0],
            [0, 0, 1, 0],
            [0, 0, 0, 1]
        ], dtype=torch.float32, device='cuda')
    )

    # Add ground plane
    builder.add_shape_plane(
        plane=(*builder.up_vector, -FLOOR_PLANE),
        width=0.0,
        length=0.0,
        cfg=SimplicitsModelBuilder.ShapeConfig(
            ke=1e4, mu=0.5, kd=100.0, kf=1.0
        ),
        label="ground_plane",
    )

    # Add collision handling between soft-body particles and shapes
    builder.add_simplicits_collisions(
        collision_particle_radius=0.1,
        detection_ratio=1.5,
        impenetrable_barrier_ratio=0.25,
        collision_penalty=1000.0,
        max_contact_pairs=10000,
        friction=0.5
    )

    # Add rigid body cube
    add_cube_mesh(builder, mesh, wp.vec3(0, 1.2, 0))

    # Configure material contact properties
    for i in range(len(builder.shape_material_ke)):
        builder.shape_material_ke[i] = 1e5
        builder.shape_material_kd[i] = 1000.0
        builder.shape_material_kf[i] = 5.0
        builder.shape_material_mu[i] = 0.5

    # Finalize model
    model = builder.finalize()
    model.soft_contact_ke = 1e5
    model.soft_contact_mu = 0.5
    model.soft_contact_kd = 1000.0
    model.soft_contact_kf = 5.0
    model.rigid_contact_ke = 1e2

    # Pre-compile the collision pipeline
    model.collide(model.state(), collision_pipeline=CollisionPipeline(model, soft_contact_margin=0.01))

    print("Model finalized")
    obj_id_0=0
    obj_id_1=1
    return model, builder, obj_id_0, obj_id_1


model, builder, obj_id_0, obj_id_1 = create_builder_model(sim_obj, mesh)
print(f"Model: {model.particle_count} particles, {model.body_count} bodies")

## States and Solvers

Newton uses a double-buffer pattern (`state_0` / `state_1`). Each solver reads from
`state_0` and writes results into `state_1`; states are then swapped.

- **`SimplicitsSolver`** — handles the two soft-body Simplicits cubes.
- **`SolverSemiImplicit`** — handles the rigid cube body.

Both solvers share the same model and write into the same state buffers.

In [ ]:
state_0 = model.state()
state_0.particle_q = model.sim_z_to_full(state_0.sim_z)
state_1 = model.state()

simplicits_solver = SimplicitsSolver(model)
rigid_solver = SolverSemiImplicit(model)

print("States and solvers ready")

## Simulation Loop

Each call to `simulate()` advances the scene by one frame:

1. **Simplicits solver** steps the soft cubes (`state_0 → state_1`), writing particle DOFs.
2. **Rigid body solver** steps the cube body (`state_0 → state_1`), writing body DOFs.
   `model.particle_count` is temporarily set to 0 so the rigid solver skips particle
   processing.
3. States are swapped for the next frame.

> **Note:** CUDA graph capture is not used here because `model.particle_count` is mutated
> on the host between the two solver calls, which is incompatible with graph capture.

In [ ]:
sim_substeps = 2
sim_dt = DT / sim_substeps


def simulate():
    global state_0, state_1

    for _ in range(sim_substeps):
        state_0.clear_forces()
        
        # Step 1: Simplicits (soft body) solver
        contacts = model.collide(state_0)
        simplicits_solver.step(state_0, state_1, None, contacts, sim_dt)
        
        # Step 2: Rigid body solver (simplicits particles disabled during this step)
        particle_count = model.particle_count
        model.particle_count = 0
        contacts = model.collide(state_0)
        rigid_solver.step(state_0, state_1, None, contacts, sim_dt)
        model.particle_count = particle_count

        state_0, state_1 = state_1, state_0

    

## Visualization with k3d

**k3d** renders all three cubes interactively in the notebook:

- **`mesh_soft`** (blue) — the two deformable Simplicits cubes **with a single affine handle each**, combined into a single k3d
  mesh by concatenating their vertices and offsetting the face indices of the second cube by
  `num_verts`. Updated each frame via `get_object_deformed_pts` (linear blend skinning).
- **`mesh_rigid`** (orange) — the single rigid body cube. Updated each frame by applying the
  body's pose (`state_0.body_q`) via `transform_points(orig_vertices_np, body_q_wp)`.

Play/Stop/Reset buttons control the simulation via a background thread.

In [ ]:
orig_vertices_np = mesh.vertices.cpu().numpy().astype(np.float32)
orig_faces_np = mesh.faces.cpu().numpy().astype(np.uint32)
orig_vertices = mesh.vertices.clone()   # keep GPU tensor for get_object_deformed_pts
num_verts = orig_vertices_np.shape[0]

# Combined mesh for the two Simplicits soft cubes
combined_faces_np = np.concatenate(
    [orig_faces_np, orig_faces_np + num_verts]
).astype(np.uint32)

plot = k3d.plot(camera_auto_fit=False)
plot.camera = [3, 3, 3,  0, 0, 0,  0, 1, 0]

mesh_soft = k3d.mesh(
    np.concatenate([orig_vertices_np, orig_vertices_np]).astype(np.float32),
    combined_faces_np,
    color=0x3399ff
)
mesh_rigid = k3d.mesh(
    orig_vertices_np.copy(),
    orig_faces_np,
    color=0xff6633
)
plot += mesh_soft
plot += mesh_rigid

# Floor plane at FLOOR_PLANE height
_s = 3.0  # half-extent
floor_verts = np.array([
    [-_s, FLOOR_PLANE, -_s],
    [ _s, FLOOR_PLANE, -_s],
    [ _s, FLOOR_PLANE, _s],
    [-_s, FLOOR_PLANE, _s],
], dtype=np.float32)
floor_faces = np.array([[0, 1, 2], [0, 2, 3]], dtype=np.uint32)
mesh_floor = k3d.mesh(floor_verts, floor_faces, color=0xcccccc, opacity=0.5)
plot += mesh_floor

plot.display()


def update_vis_mesh():
    # Soft bodies
    model.simplicits_scene.sim_z = state_0.sim_z
    v0 = model.simplicits_scene.get_object_deformed_pts(obj_id_0, orig_vertices)
    v1 = model.simplicits_scene.get_object_deformed_pts(obj_id_1, orig_vertices)
    mesh_soft.vertices = np.concatenate(
        [v0.cpu().numpy(), v1.cpu().numpy()]
    ).astype(np.float32)

    # Rigid body
    body_q = state_0.body_q.numpy()
    for i in range(len(body_q)):
        body_q_wp = wp.transform(
            wp.vec3(body_q[i][0], body_q[i][1], body_q[i][2]),
            wp.quat(body_q[i][3], body_q[i][4], body_q[i][5], body_q[i][6])
        )
        mesh_rigid.vertices = transform_points(orig_vertices_np, body_q_wp).astype(np.float32)


sim_running = [False]
sim_thread = [None]


def run_sim_loop():
    while sim_running[0]:
        simulate()
        update_vis_mesh()


def on_play(b):
    if not sim_running[0]:
        sim_running[0] = True
        sim_thread[0] = threading.Thread(target=run_sim_loop, daemon=True)
        sim_thread[0].start()


def on_stop(b):
    sim_running[0] = False


def on_reset(b):
    global state_0, state_1
    sim_running[0] = False
    if sim_thread[0] is not None:
        sim_thread[0].join()
    model.simplicits_scene.reset_scene()
    state_0 = model.state()
    state_0.particle_q = model.sim_z_to_full(state_0.sim_z)
    state_1 = model.state()
    update_vis_mesh()


buttons = [Button(description=x) for x in ["Play", "Stop", "Reset"]]
buttons[0].on_click(on_play)
buttons[1].on_click(on_stop)
buttons[2].on_click(on_reset)

update_vis_mesh()
display(VBox(buttons))